# EP4 - Guilherme Santos de Godoy

A seguir serão descritos alguns passos referentes à execução do EP4.

### Especificações do Sistema

Este exercício foi executado em um Jupyter Notebook em uma máquina virtual em meu computador pessoal. A escolha por essa opção se deu pois o sistema da máquina virtual estava melhor preparado para lidar com problemas desse tipo.

A seguir estão algumas informações referentes a esta máquina.

In [63]:
import os

num_cores = os.cpu_count()

print(f"Número de núcleos disponíveis: {num_cores}")

Número de núcleos disponíveis: 4


In [64]:
!cat /proc/cpuinfo

processor	: 0
vendor_id	: GenuineIntel
cpu family	: 6
model		: 142
model name	: Intel(R) Core(TM) i5-10210U CPU @ 1.60GHz
stepping	: 12
microcode	: 0xffffffff
cpu MHz		: 2112.006
cache size	: 6144 KB
physical id	: 0
siblings	: 4
core id		: 0
cpu cores	: 4
apicid		: 0
initial apicid	: 0
fpu		: yes
fpu_exception	: yes
cpuid level	: 22
wp		: yes
flags		: fpu vme de pse tsc msr pae mce cx8 apic sep mtrr pge mca cmov pat pse36 clflush mmx fxsr sse sse2 ht syscall nx rdtscp lm constant_tsc rep_good nopl xtopology nonstop_tsc cpuid tsc_known_freq pni pclmulqdq ssse3 cx16 pcid sse4_1 sse4_2 movbe popcnt aes rdrand hypervisor lahf_lm abm 3dnowprefetch invpcid_single ibrs_enhanced fsgsbase bmi1 bmi2 invpcid rdseed clflushopt md_clear flush_l1d arch_capabilities
bugs		: spectre_v1 spectre_v2 spec_store_bypass swapgs itlb_multihit srbds mmio_stale_data retbleed eibrs_pbrsb
bogomips	: 4224.01
clflush size	: 64
cache_alignment	: 64
address sizes	: 39 bits physical, 48 bits virtual
power management:


In [65]:
!cat /proc/meminfo

MemTotal:        4000936 kB
MemFree:          448496 kB
MemAvailable:     788488 kB
Buffers:           12816 kB
Cached:           549548 kB
SwapCached:        56680 kB
Active:          1428308 kB
Inactive:        1623368 kB
Active(anon):    1092908 kB
Inactive(anon):  1463780 kB
Active(file):     335400 kB
Inactive(file):   159588 kB
Unevictable:          16 kB
Mlocked:              16 kB
SwapTotal:       2744316 kB
SwapFree:         816876 kB
Zswap:                 0 kB
Zswapped:              0 kB
Dirty:              1128 kB
Writeback:             0 kB
AnonPages:       2449420 kB
Mapped:           285936 kB
Shmem:             67376 kB
KReclaimable:      94620 kB
Slab:             220904 kB
SReclaimable:      94620 kB
SUnreclaim:       126284 kB
KernelStack:       17920 kB
PageTables:        48668 kB
SecPageTables:         0 kB
NFS_Unstable:          0 kB
Bounce:                0 kB
WritebackTmp:          0 kB
CommitLimit:     4744784 kB
Committed_AS:   11547000 kB
VmallocTotal:   3435

In [66]:
!lsb_release -a

No LSB modules are available.
Distributor ID:	Ubuntu
Description:	Ubuntu 22.04.3 LTS
Release:	22.04
Codename:	jammy


### Execução do código sequencial

A princípio, foi feita a execução do código sequencial disponível no [repositório de referência da disciplina](https://github.com/HPCSys-Lab/HPC-101/blob/main/examples/mm-mul/mmul_seq.c).

In [67]:
%%writefile mmult_seq.c

/*
    This program multiply two N x N dense, square matrices A and B
    to yield the product matrix C = A x B
*/
#include <stdio.h>
#include <stdlib.h>
#include <time.h>
#include <sys/time.h>

// save the marix in a file
void save_matrix(double **matrix, int size){

    char file_name[30];
    sprintf(file_name, "result_matrix.txt");

    // save the result
    FILE *file;
    file = fopen(file_name, "w");

    for(int i = 0; i < size; i++){
        for(int j = 0; j < size; j++){
            fprintf(file, "%5.2f ", matrix[i][j]);
        }
        fprintf(file, "\n");
    }

    fclose(file);
}

int main(int argc, char *argv[]) {

    if(argc != 2){
        printf("Usage: ./mmul_seq N\n");
        printf("N: Matrix size on each axis\n");
        exit(-1);
    }

    // variables to measure execution time
    struct timeval time_start;
    struct timeval time_end;

    int size = atoi(argv[1]);

    // alloc memory for the matrices
    double **a = (double **) malloc(size * sizeof(double *));
    double **b = (double **) malloc(size * sizeof(double *));
    double **c = (double **) malloc(size * sizeof(double *));

    for(int i = 0; i < size; i++){
        a[i] = (double *) malloc(size * sizeof(double));
        b[i] = (double *) malloc(size * sizeof(double));
        c[i] = (double *) malloc(size * sizeof(double));
    }

    // initialize the matrices
    for(int i = 0; i < size; i++){
        for(int j = 0; j < size; j++){
            a[i][j] = i + 1;
            b[i][j] = j + 1;
        }
    }

    // get the start time
    gettimeofday(&time_start, NULL);

    // multiply a x b
    for(int i = 0; i < size; i++){
        for(int j = 0; j < size; j++){

            c[i][j] = 0;

            for(int k = 0; k < size; k++){
                c[i][j] += a[i][k] * b[k][j];
            }
        }
    }
  /*  for (int k=0; k<size; k++)
       for (int i=0; i<size; i++)
       {
          c[i][k] = 0.0;
          for (int j=0; j<size; j++)
             c[i][k] = c[i][k] + a[i][j] * b[j][k];
       }
*/
    // get the end time
    gettimeofday(&time_end, NULL);

    // save the result on a file
    save_matrix(c, size);

    double exec_time = (double) (time_end.tv_sec - time_start.tv_sec) +
                       (double) (time_end.tv_usec - time_start.tv_usec) / 1000000.0;

    printf("Matrices multiplied in %lf seconds\n", exec_time);

    return 0;
}

Writing mmult_seq.c


In [68]:
!gcc -o mmult_seq mmult_seq.c

In [69]:
!./mmult_seq 1000

Matrices multiplied in 25.839752 seconds


### Execução do código MPI

Após, a solução paralelizada utilizando MPI é apresentada e compilada.

In [76]:
%%writefile mmult_mpi.c

#include <stdio.h>
#include <stdlib.h>
#include <time.h>
#include <sys/time.h>
#include <mpi.h>

void save_matrix(double *matrix, int size, int rank) {
    char file_name[30];
    sprintf(file_name, "result_matrix_rank_%d.txt", rank);

    FILE *file;
    file = fopen(file_name, "w");

    for (int i = 0; i < size; i++) {
        for (int j = 0; j < size; j++) {
            fprintf(file, "%5.2f ", matrix[i * size + j]);
        }
        fprintf(file, "\n");
    }

    fclose(file);
}

int main(int argc, char *argv[]) {
    if (argc != 2) {
        printf("Usage: mpirun -np <num_procs> ./mmul_mpi N\n");
        printf("N: Matrix size on each axis\n");
        exit(-1);
    }

    int size, rank, num_procs;

    MPI_Init(&argc, &argv);
    MPI_Comm_size(MPI_COMM_WORLD, &num_procs);
    MPI_Comm_rank(MPI_COMM_WORLD, &rank);

    if (rank == 0) {
        printf("Number of processes: %d\n", num_procs);
    }

    struct timeval time_start;
    struct timeval time_end;

    size = atoi(argv[1]);
    int total_elements = size * size;

    double *a = (double *)malloc(total_elements * sizeof(double));
    double *b = (double *)malloc(total_elements * sizeof(double));

    if (rank == 0) {
        for (int i = 0; i < total_elements; i++) {
            a[i] = i + 1;
            b[i] = i + 1;
        }
    }

    double *c = (double *)malloc(total_elements * sizeof(double));

    MPI_Bcast(a, total_elements, MPI_DOUBLE, 0, MPI_COMM_WORLD);
    MPI_Bcast(b, total_elements, MPI_DOUBLE, 0, MPI_COMM_WORLD);

    MPI_Barrier(MPI_COMM_WORLD);
    gettimeofday(&time_start, NULL);

    int chunk_size = size / num_procs;
    
    // Alocação de uma nova matriz referente aos resultados locais. Isso se fez necessário
    // devido a um erro de "Buffers must not be aliased". Após algumas tentativas, essa acabou
    // sendo a solução que teve um resultado positivo.
    double *local_c = (double *)malloc(chunk_size * size * sizeof(double));

    for (int i = rank * chunk_size; i < (rank + 1) * chunk_size; i++) {
        for (int j = 0; j < size; j++) {
            local_c[(i - rank * chunk_size) * size + j] = 0;

            for (int k = 0; k < size; k++) {
                local_c[(i - rank * chunk_size) * size + j] += a[i * size + k] * b[k * size + j];
            }
        }
    }

    MPI_Gather(local_c, chunk_size * size, MPI_DOUBLE, c, chunk_size * size, MPI_DOUBLE, 0, MPI_COMM_WORLD);

    gettimeofday(&time_end, NULL);

    save_matrix(local_c, chunk_size, rank);

    if (rank == 0) {
        save_matrix(c, size, rank);
        double exec_time = (double)(time_end.tv_sec - time_start.tv_sec) +
                           (double)(time_end.tv_usec - time_start.tv_usec) / 1000000.0;

        printf("Matrices multiplied in %lf seconds\n", exec_time);
    }

    free(a);
    free(b);
    free(c);
    free(local_c);

    MPI_Finalize();

    return 0;
}

Overwriting mmult_mpi.c


In [71]:
!mpicc -o mmult_mpi mmult_mpi.c

### Execução do programa paralelizado

A seguir está a execução e os resultados do programa em MPI variando de 1 a 4 núcleos, conforme a disponibilidade do sistema.

In [72]:
!mpirun -np 1 ./mmult_mpi 1000

Number of processes: 1
Matrices multiplied in 21.420211 seconds


In [73]:
!mpirun -np 2 ./mmult_mpi 1000

Number of processes: 2
Matrices multiplied in 16.693640 seconds


In [74]:
!mpirun -np 3 ./mmult_mpi 1000

Number of processes: 3
Matrices multiplied in 7.797958 seconds


In [75]:
!mpirun -np 4 ./mmult_mpi 1000

Number of processes: 4
Matrices multiplied in 6.590218 seconds
